In [5]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
convert_to_fastest_fixed.py

功能：
1. 读取 user.csv, post.csv, comment.csv 三个文件（路径固定在下方变量中）
2. 将它们合并并转换为 Fastest 的输入图格式
3. 输出节点和边统计信息
4. 输出：
   - out.graph : Fastest 格式图
   - id_mapping.csv : 原始id -> 内部id, 节点类型
"""

import os
import csv
import pandas as pd
from collections import defaultdict

# ============================================================
# 🧩 用户配置区域：文件路径与输出路径
# ============================================================

BASE_DIR = "/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/structure_first"   # ⚠️ 修改成你放CSV的文件夹
USER_FILE = os.path.join(BASE_DIR, "user.csv")
POST_FILE = os.path.join(BASE_DIR, "post.csv")
COMMENT_FILE = os.path.join(BASE_DIR, "comment.csv")

OUTPUT_GRAPH = os.path.join(BASE_DIR, "parler.graph")
OUTPUT_MAP = os.path.join(BASE_DIR, "id_mapping.csv")

# ============================================================
# 🧩 类型定义
# ============================================================

NODE_LABELS = {
    "User": 0,
    "Post": 1,
    "Comment": 2
}

EDGE_LABELS = {
    "author_user_post": 0,      # user -> post (creator)
    "creator_user_comment": 1,  # user -> comment (creator)
    "post_has_comment": 2,      # post -> comment (post)
    "comment_reply_comment": 3, # comment -> comment (parent)
    "user_view_post": 4          # ✅ user -> post (view)
}

# ============================================================
# 🧩 辅助函数
# ============================================================

def clean_column_name(col: str) -> str:
    """清理CSV中的复杂列名（去掉冒号、类型标记）"""
    return col.strip().lstrip(':').split(':')[0].strip()

def load_csv_robust(path):
    """用pandas读取CSV并清理列名"""
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    new_cols = {c: clean_column_name(c) for c in df.columns}
    df.rename(columns=new_cols, inplace=True)
    return df

# ============================================================
# 🧩 主图构建逻辑
# ============================================================

def build_graph(users_df, posts_df, comments_df):
    nodes = []
    id_to_internal = {}
    internal_counter = 0

    # ---- add_node()：创建新节点 ----
    def add_node(orig_id, type_str):
        nonlocal internal_counter
        key = (type_str, orig_id)
        if key in id_to_internal:
            return id_to_internal[key]
        iid = internal_counter
        id_to_internal[key] = iid
        nodes.append((iid, orig_id, type_str, NODE_LABELS[type_str]))
        internal_counter += 1
        return iid

    # ---- 添加 User ----
    if users_df is not None:
        id_col = "id" if "id" in users_df.columns else list(users_df.columns)[0]
        for _, r in users_df.iterrows():
            add_node(str(r[id_col]), "User")

    # ---- 添加 Post ----
    if posts_df is not None:
        id_col = "id" if "id" in posts_df.columns else list(posts_df.columns)[0]
        for _, r in posts_df.iterrows():
            add_node(str(r[id_col]), "Post")

    # ---- 添加 Comment ----
    if comments_df is not None:
        id_col = "id" if "id" in comments_df.columns else list(comments_df.columns)[0]
        for _, r in comments_df.iterrows():
            add_node(str(r[id_col]), "Comment")

    # ---- 建立边集合 ----
    edges = set()

    def add_edge(type_u, orig_u, type_v, orig_v, edge_label):
        """添加一条无向边"""
        ku = (type_u, str(orig_u))
        kv = (type_v, str(orig_v))
        if ku not in id_to_internal or kv not in id_to_internal:
            return
        iu, iv = id_to_internal[ku], id_to_internal[kv]
        u, v = sorted([iu, iv])
        edges.add((u, v, edge_label))

    # ---- 生成边 (User-Post) ----
    if posts_df is not None and "creator" in posts_df.columns:
        for _, r in posts_df.iterrows():
            if r["creator"]:
                add_edge("User", r["creator"], "Post", r["id"], EDGE_LABELS["author_user_post"])

    # ---- 生成边 (User-Comment, Post-Comment, Comment-Comment) ----
    if comments_df is not None:
        for _, r in comments_df.iterrows():
            # user -> comment
            if "creator" in r and r["creator"]:
                add_edge("User", r["creator"], "Comment", r["id"], EDGE_LABELS["creator_user_comment"])
            # post -> comment
            if "post" in r and r["post"]:
                add_edge("Post", r["post"], "Comment", r["id"], EDGE_LABELS["post_has_comment"])
            # comment -> comment
            if "parent" in r and r["parent"]:
                add_edge("Comment", r["parent"], "Comment", r["id"], EDGE_LABELS["comment_reply_comment"])
            # ✅ user -> post (view)：如果该用户在该post下发了评论
            if "creator" in r and "post" in r and r["creator"] and r["post"]:
                add_edge("User", r["creator"], "Post", r["post"], EDGE_LABELS["user_view_post"])

    return nodes, edges


# ============================================================
# 🧩 输出函数
# ============================================================

def write_fastest_graph(out_path, nodes, edges):
    """写入Fastest格式"""
    deg = defaultdict(int)
    for u, v, _ in edges:
        deg[u] += 1
        deg[v] += 1

    with open(out_path, "w") as fout:
        fout.write(f"t {len(nodes)} {len(edges)}\n")
        for iid, orig_id, type_str, label_int in sorted(nodes, key=lambda x: x[0]):
            fout.write(f"v {iid} {label_int} {deg.get(iid, 0)}\n")
        for u, v, e in sorted(edges):
            fout.write(f"e {u} {v} {e}\n")

def write_mapping(map_path, nodes):
    """保存id映射"""
    with open(map_path, "w", newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["internal_id", "orig_id", "type", "label_int"])
        for iid, orig_id, type_str, label_int in sorted(nodes, key=lambda x: x[0]):
            writer.writerow([iid, orig_id, type_str, label_int])

def count_statistics(nodes, edges):
    """统计节点和边数量"""
    node_count = defaultdict(int)
    for _, _, t, _ in nodes:
        node_count[t] += 1

    edge_count = defaultdict(int)
    for _, _, e in edges:
        edge_count[e] += 1

    print("\n===== 📊 图统计信息 =====")
    print(f"总节点数: {len(nodes)}")
    for t, cnt in node_count.items():
        print(f"  {t:<10}: {cnt}")
    print(f"总边数: {len(edges)}")
    for e, cnt in edge_count.items():
        relation = [k for k, v in EDGE_LABELS.items() if v == e]
        print(f"  EdgeLabel {e} ({relation[0] if relation else '未知'}): {cnt}")


# ============================================================
# 🧩 主程序入口
# ============================================================

def main():
    # 1️⃣ 读取数据
    users_df = load_csv_robust(USER_FILE) if os.path.exists(USER_FILE) else None
    posts_df = load_csv_robust(POST_FILE) if os.path.exists(POST_FILE) else None
    comments_df = load_csv_robust(COMMENT_FILE) if os.path.exists(COMMENT_FILE) else None

    # 2️⃣ 构建图
    nodes, edges = build_graph(users_df, posts_df, comments_df)
    print(f"[INFO] 已构建图: 节点数={len(nodes)}, 边数={len(edges)}")

    # 3️⃣ 写出图文件
    write_fastest_graph(OUTPUT_GRAPH, nodes, edges)
    write_mapping(OUTPUT_MAP, nodes)
    print(f"[INFO] 图文件已保存到: {OUTPUT_GRAPH}")
    print(f"[INFO] 映射文件已保存到: {OUTPUT_MAP}")

    # 4️⃣ 输出统计信息
    count_statistics(nodes, edges)


if __name__ == "__main__":
    main()


[INFO] 已构建图: 节点数=175688, 边数=337788
[INFO] 图文件已保存到: /home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/structure_first/parler.graph
[INFO] 映射文件已保存到: /home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/structure_first/id_mapping.csv

===== 📊 图统计信息 =====
总节点数: 175688
  User      : 41342
  Post      : 24082
  Comment   : 110264
总边数: 337788
  EdgeLabel 2 (post_has_comment): 110264
  EdgeLabel 1 (creator_user_comment): 110264
  EdgeLabel 4 (user_view_post): 93178
  EdgeLabel 0 (author_user_post): 24082
